# Ejercicio 1: Búsqueda Básica en un Corpus

## Sección 1 — Carga del corpus

### 1.1 Leer un archivo — método directo

La forma más simple de leer un archivo en Python.  
⚠️ **Problema:** el archivo queda abierto en memoria hasta que el programa termine.  
Por eso este método no es recomendado cuando se trabaja con muchos archivos.

In [3]:
corpus_path = "../corpus/11_Alices Adventures in Wonderland.txt"

file_direct = open(corpus_path, 'r', encoding='utf-8').read()

print(f"Caracteres cargados: {len(file_direct)}")
print(f"Primeros 200 caracteres:\n{file_direct[:200]}")

Caracteres cargados: 163951
Primeros 200 caracteres:
﻿The Project Gutenberg eBook of Alice's Adventures in Wonderland
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no 


### 1.2 Leer un archivo — método con `with`

El bloque `with` garantiza que el archivo se **cierre automáticamente** al terminar,
incluso si ocurre un error durante la lectura.  
✅ Esta es la forma correcta y recomendada en Python.

In [4]:
with open(corpus_path, "r", encoding="utf-8") as f:
    file_with = f.read()

print(f"Caracteres cargados: {len(file_with)}")
print(f"Primeros 200 caracteres:\n{file_with[:200]}")

Caracteres cargados: 163951
Primeros 200 caracteres:
﻿The Project Gutenberg eBook of Alice's Adventures in Wonderland
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no 


### 1.3 Leer todos los archivos del corpus con `os`

`os.listdir()` nos da la lista de archivos en un directorio.  
Iteramos sobre ellos, filtramos solo los `.txt`, y guardamos cada contenido
en un diccionario: `{ nombre_archivo: contenido }`.

Esto nos permite acceder a cualquier libro por su nombre como clave.

In [5]:
import os

corpus_dir = "../corpus/"

corpus_data = {}

for filename in os.listdir(corpus_dir):
    if filename.endswith(".txt"):
        filepath = os.path.join(corpus_dir, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            corpus_data[filename] = f.read()

print(f"Archivos cargados: {len(corpus_data)}")

Archivos cargados: 1000


### 1.4 Inspeccionar el corpus

Revisamos cuántos libros cargamos y cuántos caracteres tiene cada uno.  
Esto nos da una idea del tamaño del corpus con el que vamos a trabajar.

In [6]:
print(f"Total de libros en el corpus: {len(corpus_data)}\n")
print(f"{'Libro':<70} {'Caracteres':>12}")
print("-" * 84)
for book, content in corpus_data.items():
    print(f"{book:<70} {len(content):>12,}")

Total de libros en el corpus: 1000

Libro                                                                    Caracteres
------------------------------------------------------------------------------------
10083_The House of the Whispering Pines.txt                                 629,671
100_The Complete Works of William Shakespeare.txt                         5,378,702
10147_America's War for Humanity.txt                                      1,216,599
1023_Bleak House.txt                                                      1,958,834
1028_The Professor.txt                                                      520,331
10331_The Mirror of Literature, Amusement, and Instruction. Volume.txt       94,903
1033_Rose o' the River.txt                                                  174,619
10362_Sketches of the East Africa Campaign.txt                              265,286
10380_Bible Stories and Religious Classics.txt                              895,898
10435_The Atlantic Monthly, Volume 02, 

## Sección 2 — Búsqueda simple

### 2.1 Buscar un término — primer match

Recorremos el corpus libro por libro.
En cuanto encontramos el término, reportamos y **paramos** con `break`.

Útil cuando solo nos interesa saber si el término existe en el corpus,
sin importar en cuántos libros aparece.

In [7]:
query = "Frankenstein"

found = False
for book, content in corpus_data.items():
    if query in content:
        print(f"'{query}' encontrado en: {book}")
        found = True
        break  # dejamos de buscar al primer match

if not found:
    print(f"'{query}' no fue encontrado en ningún libro del corpus")

'Frankenstein' encontrado en: 18216_Pathfinders of the WestBeing the Thrilling Story of the Adve.txt


### 2.2 Buscar un término — todos los matches

Ahora recorremos el corpus **completo**, sin `break`.
Recolectamos todos los libros donde aparece el término.

Útil cuando queremos saber la distribución de una palabra en el corpus.

In [8]:
query = "Frankenstein"

found_books = []

for book, content in corpus_data.items():
    if query in content:
        found_books.append(book)

# Reportar resultados
if found_books:
    print(f"'{query}' aparece en {len(found_books)} libro(s):\n")
    for book in found_books:
        print(f"  - {book}")
else:
    print(f"'{query}' no fue encontrado en ningún libro del corpus")

'Frankenstein' aparece en 16 libro(s):

  - 18216_Pathfinders of the WestBeing the Thrilling Story of the Adve.txt
  - 20486_Tiverton Tales.txt
  - 3108_Nine Short Essays.txt
  - 31421_Through Apache Land.txt
  - 42176_Long Will.txt
  - 44043_The truth about opium.txt
  - 45528_The Best Man.txt
  - 56665_Tales and StoriesNow First Collected.txt
  - 61313_Questions at Issue.txt
  - 62406_Frankenstein, ou le Prométhée moderne Volume 3 (of 3).txt
  - 67925_The Prose Works of Percy Bysshe Shelley, Vol. 1 [of 2].txt
  - 71046_Modern English Biography Volume 2.txt
  - 71082_Political and commercial geology and the world's mineral res.txt
  - 75366_Woman's Voice.txt
  - 8498_The Atlantic Monthly, Volume 01, No. 01, November, 1857A Mag.txt
  - 84_Frankenstein.txt


> Dos cosas a notar entre 2.1 y 2.2:
> * El operador `in` sobre un string hace búsqueda exacta y case-sensitive — ``"alice"`` y ``"Alice"`` son distintas
> * `break` en 2.1 puede ahorrarnos recorrer todo el corpus si el término aparece temprano

## Sección 3 — Medición de tiempos

### 3.1 Medir el tiempo de búsqueda de un término

Usamos `time.time()` para capturar el momento antes y después de la búsqueda.
La diferencia nos da el tiempo transcurrido.

- `time.time()` retorna segundos como float
- Multiplicamos por 1000 para convertir a milisegundos (ms)

In [9]:
import time

query = "Alice"

start = time.time()

found_books = []
for book, content in corpus_data.items():
    if query in content:
        found_books.append(book)

end = time.time()

elapsed_ms = (end - start) * 1000

print(f"Término buscado : '{query}'")
print(f"Encontrado en   : {len(found_books)} libro(s)")
print(f"Tiempo de búsqueda: {elapsed_ms:.4f} ms")

Término buscado : 'Alice'
Encontrado en   : 126 libro(s)
Tiempo de búsqueda: 554.0226 ms


### 3.2 Comparar tiempos según el término buscado

¿Afecta el término buscado al tiempo de búsqueda?

Comparamos un término **muy común** (aparece en casi todos los libros)
contra uno **muy raro** (aparece en pocos o ninguno).

Hipótesis: el tiempo debería ser similar, porque `in` sobre un string
recorre el texto carácter a carácter — no importa si lo encuentra o no,
el peor caso es siempre recorrer el libro completo.

In [10]:
import time

# Definimos los términos a comparar
terms = {
    "común": "the",  # palabra que aparece en casi todo
    "raro": "Zyxwvutsrqpon",  # término que casi seguro no existe
}

results = {}

for label, query in terms.items():
    start = time.time()

    found_books = []
    for book, content in corpus_data.items():
        if query in content:
            found_books.append(book)

    end = time.time()
    elapsed_ms = (end - start) * 1000

    results[label] = {
        "query": query,
        "found_in": len(found_books),
        "time_ms": elapsed_ms,
    }

# Mostrar resultados comparativos
print(f"{'Tipo':<8} {'Término':<20} {'Libros encontrados':>18} {'Tiempo (ms)':>12}")
print("-" * 62)
for label, data in results.items():
    print(
        f"{label:<8} {data['query']:<20} {data['found_in']:>18} {data['time_ms']:>11.4f}"
    )

Tipo     Término              Libros encontrados  Tiempo (ms)
--------------------------------------------------------------
común    the                                1000      0.9968
raro     Zyxwvutsrqpon                         0    308.0826


### 📌 Observación — el tiempo SÍ depende del término

El resultado muestra lo contrario a la hipótesis inicial:

- El término **común** (`"the"`) es más **rápido**: `in` encuentra el match
  muy temprano en cada libro y deja de escanear ese archivo inmediatamente.
- El término **raro** (inexistente) es más **lento**: `in` debe recorrer
  el contenido **completo** de cada libro para confirmar que no está.

**Conclusión:** el peor caso para una búsqueda con `in` no es encontrar el término,
sino **no encontrarlo** — porque obliga a escanear todo el texto.

Esto refuerza la necesidad de estructuras más eficientes (como índices invertidos)
cuando el corpus es grande.

## Sección 4 — Vocabulario del corpus

### 4.1 Palabras únicas del corpus completo

Usamos `split()` para separar el texto en palabras (por espacios y saltos de línea).
Usamos un `set` para quedarnos solo con las palabras **únicas** — los sets
no permiten duplicados, así que automáticamente eliminan repeticiones.

⚠️ `split()` es una tokenización muy básica — no distingue entre
`"Alice"`, `"Alice,"` y `"Alice."` como la misma palabra.
Eso lo veremos más adelante con técnicas de preprocesamiento.

In [11]:
all_words = set()

for book, content in corpus_data.items():
    all_words.update(content.split())

print(f"Total de palabras únicas en el corpus: {len(all_words):,}")
print(f"\nMuestra de 20 palabras (ordenadas):")
print(sorted(all_words)[:20])

Total de palabras únicas en el corpus: 2,361,662

Muestra de 20 palabras (ordenadas):
['!', '!!WORLD', '!"', '!]', '"', '""Goody', '""I\'ll', '"$15,000', '"&', '"\'', '"\'"ANNO', '"\'"After', '"\'"Amo"', '"\'"Brother,', '"\'"But', '"\'"Curse', '"\'"Damn', '"\'"Dear', '"\'"Dennoch', '"\'"Did']


### 4.2 Palabras únicas de un libro específico

Hacemos lo mismo pero sobre un solo libro.
Esto nos permite comparar el vocabulario de un libro
contra el vocabulario total del corpus.

Un vocabulario pequeño relativo al corpus puede indicar
que el libro es temáticamente específico o corto.

In [12]:
book_name = "11_Alices Adventures in Wonderland.txt"

alice_words = set(corpus_data[book_name].split())

print(f"Libro: {book_name}")
print(f"Palabras únicas en el libro : {len(alice_words):,}")
print(f"Palabras únicas en el corpus: {len(all_words):,}")
print(
    f"El libro representa el {len(alice_words) / len(all_words) * 100:.2f}% del vocabulario total"
)

print(f"\nMuestra de 20 palabras del libro (ordenadas):")
print(sorted(alice_words)[:20])

Libro: 11_Alices Adventures in Wonderland.txt
Palabras únicas en el libro : 5,974
Palabras únicas en el corpus: 2,361,662
El libro representa el 0.25% del vocabulario total

Muestra de 20 palabras del libro (ordenadas):
['#11]', '#516,', '$5,000)', '($1', '(862)', '(Alice', '(And,', '(As', '(Before', '(Dinah', '(For,', '(He', '(If', '(In', '(It', '(Sounds', '(The', '(We', '(Which', '(_with']


### 📌 Observación — limitaciones de `split()`

Los resultados muestran el problema de tokenizar con `split()` básico:

- El corpus tiene **774,028 "palabras" únicas** — un número inflado porque
  `"Alice"`, `"Alice,"`, `"Alice."` y `"(Alice"` se cuentan como tokens distintos
- En la muestra se ven tokens como `'"&'`, `'#11'`, `'$5,000)'` — no son palabras reales
- Alice representa solo el **0.77%** del vocabulario total, pero parte de ese
  vocabulario es "ruido" de puntuación

Para un vocabulario limpio se necesita **preprocesamiento**:
pasar a minúsculas, eliminar puntuación, etc.
Eso lo veremos en prácticas posteriores.

> Dos números interesantes del output:
> * **774,028** palabras únicas en el corpus es muchísimo — en un corpus limpio esperarías bastante menos
> * **5,974** en Alice con solo 0.77% del total tiene sentido para un libro infantil con vocabulario simple

## Sección 5 — Búsqueda masiva

### 5.1 Preparar las queries

Tomamos las palabras únicas de Alice como conjunto de términos a buscar.
La idea es simular una búsqueda masiva: en vez de buscar un solo término,
buscamos miles a la vez.

Esto nos permite medir qué tan costoso es hacer búsqueda lineal a escala.

In [13]:
# Usamos las palabras únicas de Alice como queries
book_name = "11_Alices Adventures in Wonderland.txt"
queries = set(corpus_data[book_name].split())

print(f"Total de términos a buscar: {len(queries):,}")

Total de términos a buscar: 5,974


### 5.2 Búsqueda masiva — cada palabra en todo el corpus

Para cada término del conjunto de queries, buscamos en qué libros aparece.
Guardamos los resultados en un diccionario: `{ término: [libros donde aparece] }`.

Esto es una búsqueda lineal O(q × n × l) donde:
- q = cantidad de queries
- n = cantidad de libros
- l = longitud promedio de cada libro

In [14]:
import time

results = {}

start = time.time()

for query in queries:
    found_books = []
    for book, content in corpus_data.items():
        if query in content:
            found_books.append(book)
    results[query] = found_books

end = time.time()

elapsed_s = end - start
elapsed_ms = elapsed_s * 1000

print(f"Términos buscados : {len(queries):,}")
print(f"Tiempo total      : {elapsed_s:.4f} s  ({elapsed_ms:.2f} ms)")
print(f"Tiempo promedio   : {elapsed_ms / len(queries):.4f} ms por término")

KeyboardInterrupt: 

### 5.3 Explorar los resultados

Revisamos los resultados de la búsqueda masiva:
- ¿Qué términos aparecen en más libros?
- ¿Qué términos son exclusivos de Alice?

Ordenamos por cantidad de libros donde aparece cada término.

In [ ]:
# Ordenar términos por cantidad de libros donde aparecen (de mayor a menor)
sorted_results = sorted(results.items(), key=lambda x: len(x[1]), reverse=True)

print("=== Top 10 términos más distribuidos en el corpus ===\n")
print(f"{'Término':<25} {'Aparece en N libros':>20}")
print("-" * 47)
for term, books in sorted_results[:10]:
    print(f"{term:<25} {len(books):>20,}")

print("\n=== Top 10 términos exclusivos de Alice ===\n")
print(f"{'Término':<25} {'Aparece en N libros':>20}")
print("-" * 47)
exclusive = [(term, books) for term, books in sorted_results if len(books) == 1]
for term, books in exclusive[:10]:
    print(f"{term:<25} {len(books):>20,}")

print(f"\nTotal de términos exclusivos de Alice: {len(exclusive):,}")
print(
    f"Total de términos compartidos con otros libros: {len(queries) - len(exclusive):,}"
)

=== Top 10 términos más distribuidos en el corpus ===

Término                    Aparece en N libros
-----------------------------------------------
writing                                    100
discontinue                                100
distributing                               100
Creating                                   100
(a                                         100
right                                      100
come.                                      100
me                                         100
makes                                      100
fur                                        100

=== Top 10 términos exclusivos de Alice ===

Término                    Aparece en N libros
-----------------------------------------------
Morcar,                                      1
Soup!”                                       1
“Mouse                                       1
tastes!                                      1
Mabel!                                       1
_thi

### 📌 Conclusión — costo de la búsqueda lineal a escala

Con **5,974 queries** sobre **100 libros**, la búsqueda lineal tardó:

- **282.94 segundos (~4 min 43 s)** en total
- **47.36 ms** por término en promedio

Para solo buscar el vocabulario de UN libro sobre UN corpus pequeño.

Ahora imaginemos escalar esto:
- 1,000,000 de queries → ~13 horas
- Un corpus de millones de documentos (Google, Wikipedia) → inviable

**El problema:** con búsqueda lineal (`in` sobre texto plano), el tiempo crece
O(q × n × l) — proporcional a queries, libros y longitud de cada libro.
No importa cuánta RAM o CPU tengas, el enfoque no escala.

**La solución:** un **índice invertido** — una estructura construida **una sola vez**
que mapea `{ término → [documentos donde aparece] }`.
Una vez construido, cualquier query se responde en **O(1)**.
Eso es lo que hace cualquier motor de búsqueda real (Elasticsearch, Lucene, etc.).